In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/chinmay/Programming/annealedRL/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "Qwen/Qwen3-0.6B"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": ""}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)

In [62]:
sample_thinking_trace = "ok so I'm thinking about the user's request. I'm going to give a short introduction to large language model that starts with"

In [63]:
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).split("\n</think>\n")[0] + sample_thinking_trace
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [64]:
outputs = model.generate(**model_inputs, max_new_tokens=16000)

In [67]:
print(tokenizer.decode(outputs[0]))

<|im_start|>user
Give me a short introduction to large language model.<|im_end|>
<|im_start|>assistant
<think>
ok so I'm thinking about the user's request. I'm going to give a short introduction to large language model that starts with a brief explanation of what it is, its purpose, and some key features. I should keep it concise and engaging, making sure to highlight the benefits and applications of such models.
</think>

A large language model (LLM) is a type of artificial intelligence designed to understand, generate, and respond to human language. These models are trained on vast amounts of text data to learn patterns and understand context, allowing them to perform a wide range of tasks, from answering questions to writing creative content. They are used in various applications such as customer service, content creation, and language translation.<|im_end|>


#### Annealing for Inference engine

- Generations for user to system tokens
- add in the annealing functions, which will anneal on tokens give on the inference side
- Input from the dataset that supports annleaing (dataloader side)
- instead of samples, reutrn back Prompt samples like a wrapper, at run time
- fix the logic up in train and inference
- make a scheduler for this lr, self.global_step which can be passed in

- how we parse the reward, look at the utils in the messages
- start experiments with the lr ablation for scheduling the annealing
- make sure all the pipelines work with this